# Model Experiment: LightGBM

This notebook mirrors `model_experiment_XGBoost.ipynb`, using the same `WalmartBasePreprocessor` and `WalmartTabularFeatureEngineer` classes, so results are directly comparable between the two tree-based architectures.

LightGBM differs from XGBoost in how it grows trees (leaf-wise instead of level-wise) and in its native handling of missing values. Because of this, we also test an alternative feature set using the raw (NaN-preserved) markdown columns instead of the 0-filled + indicator version, to see if LightGBM's built-in missing-value handling performs better on its own

In [3]:
import dagshub
import mlflow
import lightgbm as lgb
import numpy as np
import pandas as pd
import optuna
import sys

sys.path.append("..")

from src.data.load_data import load_raw_data
from src.data.splits import last_n_weeks_split
from src.features.base import WalmartBasePreprocessor
from src.features.tabular import WalmartTabularFeatureEngineer

dagshub.init(repo_owner="LukaBatilashvili07", repo_name="walmart-sales-forecasting", mlflow=True)
mlflow.set_experiment("LightGBM_Training")
mlflow.lightgbm.autolog(log_models=False)

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

## 1. Data Loading and Train/Validation Split

Same split as XGBoost: the last 39 weeks of `train.csv` held out for validation, matching the length of the real `test.csv`

In [4]:
train_raw, test_raw, stores, features = load_raw_data("../data/raw")

train_part, valid_part = last_n_weeks_split(train_raw, n_weeks=39)

print(train_part["Date"].min(), "-", train_part["Date"].max())
print(valid_part["Date"].min(), "-", valid_part["Date"].max())

2010-02-05 00:00:00 - 2012-01-27 00:00:00
2012-02-03 00:00:00 - 2012-10-26 00:00:00


## 2. Run: LightGBM_Cleaning

Identical preprocessing to the XGBoost notebook, same `WalmartBasePreprocessor`, fit only on `stores`/`features`

In [5]:
with mlflow.start_run(run_name="LightGBM_Cleaning"):
    base_pre = WalmartBasePreprocessor()
    base_pre.fit(stores, features)

    train_base = base_pre.transform(train_part)
    valid_base = base_pre.transform(valid_part)

    mlflow.log_param("markdown_start_date", str(base_pre.MARKDOWN_START_DATE.date()))
    mlflow.log_metric("train_rows", len(train_base))
    mlflow.log_metric("valid_rows", len(valid_base))
    mlflow.log_metric("train_nan_total", int(train_base.isna().sum().sum()))
    mlflow.log_metric("valid_nan_total", int(valid_base.isna().sum().sum()))

    train_base.to_parquet("_cache_train_base_lgb.parquet")
    valid_base.to_parquet("_cache_valid_base_lgb.parquet")

🏃 View run LightGBM_Cleaning at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3/runs/7adacb71ddbd45f5994156a8597eff76
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3


## 3. Run: LightGBM_Feature_Selection

Same feature set as XGBoost, plus one addition: the raw markdown columns are tested alongside the 0-filled versions. LightGBM can split on missing values natively

In [6]:
train_base = pd.read_parquet("_cache_train_base_lgb.parquet")
valid_base = pd.read_parquet("_cache_valid_base_lgb.parquet")

with mlflow.start_run(run_name="LightGBM_Feature_Selection"):
    tab_eng = WalmartTabularFeatureEngineer()
    tab_eng.fit(train_base[["Store", "Dept", "Date", "Weekly_Sales"]])

    train_full = tab_eng.transform_train(train_base)
    valid_full = tab_eng.transform_future(valid_base)

    feature_cols = [
        "Store", "Dept", "Size", "Temperature", "Fuel_Price",
        "CPI", "Unemployment",
        "Type_encoded",
        "Year", "Month", "WeekOfYear", "Week_sin", "Week_cos",
        "IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas",
        "MarkDown1_raw", "MarkDown2_raw", "MarkDown3_raw", "MarkDown4_raw", "MarkDown5_raw",
        "has_markdown_signal", "markdown_missing_count", "markdown_available_period",
        "Sales_lag_1", "Sales_lag_4", "Sales_lag_52",
        "Sales_roll_mean_4", "Sales_roll_std_4", "Sales_roll_mean_12",
    ]

    X_train, y_train = train_full[feature_cols], train_full["Weekly_Sales"]
    X_valid, y_valid = valid_full[feature_cols], valid_full["Weekly_Sales"]

    baseline = lgb.LGBMRegressor(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
    baseline.fit(X_train, y_train)

    importances = pd.Series(baseline.feature_importances_, index=feature_cols).sort_values(ascending=False)
    preds = baseline.predict(X_valid)
    baseline_wmae = wmae(y_valid.values, preds, valid_full["IsHoliday"].values)

    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("markdown_variant", "raw_nan")
    mlflow.log_metric("baseline_wmae", baseline_wmae)
    importances.to_frame("importance").to_csv("_feature_importance_lgb.csv")
    mlflow.log_artifact("_feature_importance_lgb.csv")

    print(importances.head(15))
    print("Baseline WMAE:", baseline_wmae)

    train_full.to_parquet("_cache_train_full_lgb.parquet")
    valid_full.to_parquet("_cache_valid_full_lgb.parquet")

2026/07/11 17:01:43 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008617 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4118
[LightGBM] [Info] Number of data points in the train set: 305982, number of used features: 31
[LightGBM] [Info] Start training from score 16033.559487
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

2026/07/11 17:01:51 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Sales_lag_52          1038
Sales_roll_std_4       999
Sales_lag_1            942
Dept                   848
WeekOfYear             639
Sales_lag_4            525
Sales_roll_mean_4      464
Sales_roll_mean_12     417
Fuel_Price             366
Week_sin               279
IsThanksgiving         258
Temperature            250
MarkDown3_raw          235
Size                   225
Week_cos               223
dtype: int32
Baseline WMAE: 5463.721763029374
🏃 View run LightGBM_Feature_Selection at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3/runs/786d6092397642aabc53f7332dd20e99
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3


## 4. Run: LightGBM_CV Walk-Forward Cross-Validation

Same walk-forward logic as the XGBoost notebook: three folds, each with a growing training window and a 13-week validation window right after it. This checks whether LightGBM's performance is stable across time periods, using the same fold boundaries as XGBoost for comparison

In [7]:
train_base = pd.read_parquet("_cache_train_base_lgb.parquet")

def make_walk_forward_folds(dates, n_folds=3, val_weeks=13):
    dates = sorted(dates)
    folds = []
    for i in range(n_folds, 0, -1):
        cutoff = len(dates) - i * val_weeks
        if cutoff <= 0:
            continue
        train_dates = dates[:cutoff]
        val_dates = dates[cutoff:cutoff + val_weeks]
        folds.append((train_dates, val_dates))
    return folds

unique_dates = train_base["Date"].unique()
folds = make_walk_forward_folds(unique_dates, n_folds=3, val_weeks=13)

with mlflow.start_run(run_name="LightGBM_CV"):
    fold_scores = []

    for i, (train_dates, val_dates) in enumerate(folds):
        fold_train_raw = train_base[train_base["Date"].isin(train_dates)]
        fold_val_raw = train_base[train_base["Date"].isin(val_dates)]

        fold_tab_eng = WalmartTabularFeatureEngineer()
        fold_tab_eng.fit(fold_train_raw[["Store", "Dept", "Date", "Weekly_Sales"]])

        fold_train_full = fold_tab_eng.transform_train(fold_train_raw)
        fold_val_full = fold_tab_eng.transform_future(fold_val_raw)

        X_tr = fold_train_full[feature_cols]
        y_tr = fold_train_full["Weekly_Sales"]
        X_val = fold_val_full[feature_cols]
        y_val = fold_val_full["Weekly_Sales"]

        model = lgb.LGBMRegressor(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        score = wmae(y_val.values, preds, fold_val_full["IsHoliday"].values)

        fold_scores.append(score)
        mlflow.log_metric(f"fold_{i}_wmae", score)
        print(f"Fold {i}: train up to {train_dates[-1]}, val {val_dates[0]}-{val_dates[-1]}, WMAE={score:.2f}")

    mlflow.log_metric("cv_wmae_mean", float(np.mean(fold_scores)))
    mlflow.log_metric("cv_wmae_std", float(np.std(fold_scores)))
    print("CV WMAE mean/std:", np.mean(fold_scores), np.std(fold_scores))

2026/07/11 17:30:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004483 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2799
[LightGBM] [Info] Number of data points in the train set: 190674, number of used features: 23
[LightGBM] [Info] Start training from score 15968.639532
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

2026/07/11 17:31:00 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fold 0: train up to 2011-04-29 00:00:00, val 2011-05-06 00:00:00-2011-07-29 00:00:00, WMAE=5939.96


2026/07/11 17:31:08 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


...

Fold 1: train up to 2011-07-29 00:00:00, val 2011-08-05 00:00:00-2011-10-28 00:00:00, WMAE=6075.09


2026/07/11 17:31:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003835 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2857
[LightGBM] [Info] Number of data points in the train set: 267184, number of used features: 23
[LightGBM] [Info] Start training from score 15864.893370
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

2026/07/11 17:31:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fold 2: train up to 2011-10-28 00:00:00, val 2011-11-04 00:00:00-2012-01-27 00:00:00, WMAE=8213.54
CV WMAE mean/std: 6742.862334572361 1041.3901650359896
🏃 View run LightGBM_CV at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3/runs/d643af6376174eb8899779d3d6bbd3f7
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3


## 5. Run: LightGBM_HPO

Optuna searches LightGBM-specific parameters, most importantly num_leaves, which controls tree complexity in leaf growth. The objective is WMAE, same as XGBoost, for comparison. 

In [8]:
train_full = pd.read_parquet("_cache_train_full_lgb.parquet")
valid_full = pd.read_parquet("_cache_valid_full_lgb.parquet")

X_train, y_train = train_full[feature_cols], train_full["Weekly_Sales"]
X_valid, y_valid = valid_full[feature_cols], valid_full["Weekly_Sales"]
holiday_valid = valid_full["IsHoliday"].values

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 5.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
    }

    with mlflow.start_run(run_name=f"LightGBM_HPO_trial_{trial.number}", nested=True):
        model = lgb.LGBMRegressor(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_valid)
        score = wmae(y_valid.values, preds, holiday_valid)

        mlflow.log_params(params)
        mlflow.log_metric("wmae", score)

    return score

with mlflow.start_run(run_name="LightGBM_HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=30)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_wmae", study.best_value)

best_params = study.best_params
print("Best parameters:", best_params)
print("Best WMAE:", study.best_value)

[I 2026-07-11 18:29:52,322] A new study created in memory with name: no-name-54f9d955-b4fe-419a-9822-5d16ddee1e1e
2026/07/11 18:29:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013656 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4118
[LightGBM] [Info] Number of data points in the train set: 305982, number of used features: 31
[LightGBM] [Info] Start training from score 16033.559487
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

2026/07/11 18:30:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


🏃 View run LightGBM_HPO_trial_0 at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3/runs/d74c230d5164476ba7ce95743de6f316
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3


[I 2026-07-11 18:30:12,452] Trial 0 finished with value: 5781.537790454874 and parameters: {'n_estimators': 300, 'num_leaves': 209, 'max_depth': 9, 'learning_rate': 0.012469006023328465, 'subsample': 0.6632517735808194, 'colsample_bytree': 0.9792662138283859, 'min_child_samples': 81, 'reg_lambda': 0.2519956567532428}. Best is trial 0 with value: 5781.537790454874.
2026/07/11 18:30:16 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can dec

...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008453 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4118
[LightGBM] [Info] Number of data points in the train set: 305982, number of used features: 31
[LightGBM] [Info] Start training from score 16033.559487
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

2026/07/11 19:05:29 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


🏃 View run LightGBM_HPO_trial_29 at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3/runs/2ed140dfd458407dba88789678e5d188
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3


[I 2026-07-11 19:05:31,703] Trial 29 finished with value: 5046.6387459168645 and parameters: {'n_estimators': 800, 'num_leaves': 145, 'max_depth': 9, 'learning_rate': 0.013722051033470437, 'subsample': 0.8765689445592314, 'colsample_bytree': 0.791618230139984, 'min_child_samples': 42, 'reg_lambda': 0.19965031663222851}. Best is trial 18 with value: 4891.0937453407005.


🏃 View run LightGBM_HPO at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3/runs/bb15be02d78049f69445c946c9ea4f37
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3
Best parameters: {'n_estimators': 700, 'num_leaves': 138, 'max_depth': 8, 'learning_rate': 0.027286640400932736, 'subsample': 0.9583676748478898, 'colsample_bytree': 0.6426299694944394, 'min_child_samples': 70, 'reg_lambda': 0.27439413292941295}
Best WMAE: 4891.0937453407005


## 6. Run: LightGBM_Best Final Model as a Pipeline

Same as the XGBoost pipeline: `WalmartLGBMPipeline` wraps fitted `base_preprocessor`, `tabular_engineer`, and trained LightGBM model in one object, so it can take a raw test set and return predictions.

In [9]:
class WalmartLGBMPipeline(mlflow.pyfunc.PythonModel):
    def __init__(self, base_preprocessor, tabular_engineer, model, feature_cols):
        self.base_preprocessor = base_preprocessor
        self.tabular_engineer = tabular_engineer
        self.model = model
        self.feature_cols = feature_cols

    def predict(self, context, model_input: pd.DataFrame):
        base = self.base_preprocessor.transform(model_input)
        full = self.tabular_engineer.transform_future(base)
        X = full[self.feature_cols]
        return self.model.predict(X)


with mlflow.start_run(run_name="LightGBM_Best") as run:
    final_model = lgb.LGBMRegressor(**best_params, random_state=42, n_jobs=-1)
    final_model.fit(X_train, y_train)

    preds = final_model.predict(X_valid)
    final_wmae = wmae(y_valid.values, preds, holiday_valid)
    mlflow.log_params(best_params)
    mlflow.log_metric("final_valid_wmae", final_wmae)
    print("Final validation WMAE:", final_wmae)

    pipeline = WalmartLGBMPipeline(
        base_preprocessor=base_pre,
        tabular_engineer=tab_eng,
        model=final_model,
        feature_cols=feature_cols,
    )

    mlflow.pyfunc.log_model(
        artifact_path="lightgbm_pipeline",
        python_model=pipeline,
        registered_model_name="Walmart_LightGBM_Pipeline",
    )

2026/07/11 19:10:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007539 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4118
[LightGBM] [Info] Number of data points in the train set: 305982, number of used features: 31
[LightGBM] [Info] Start training from score 16033.559487
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

2026/07/11 19:10:35 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/07/11 19:10:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final validation WMAE: 4891.0937453407005


2026/07/11 19:10:38 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.
2026/07/11 19:10:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'Walmart_LightGBM_Pipeline'.
2026/07/11 19:11:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_LightGBM_Pipeline, version 1
Created version '1' of model 'Walmart_LightGBM_Pipeline'.


🏃 View run LightGBM_Best at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3/runs/49ea180f7d5e48ac84c533d9e07c036a
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3


### Verifying the Pipeline Works on Raw Data

Same check as XGBoost: load the pipeline back by run ID and call `.predict()` directly on `test_raw`, the untouched test dataframe.

In [10]:
loaded = mlflow.pyfunc.load_model("models:/Walmart_LightGBM_Pipeline/1")
test_preds = loaded.predict(test_raw)
print(test_preds[:10])

2026/07/11 19:11:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[21963.63732713 25029.88863101 26546.19594783 19797.03250777
 23060.76623163 25184.18779184 24166.93173959 25102.62979558
 18281.97567524 24647.75955443]


## 7. Run: LightGBM_Final_Refit
Refit the best-params model on the ENTIRE train.csv (all 143 weeks), not just train_part (104 weeks)

In [11]:
with mlflow.start_run(run_name="LightGBM_Final_Refit") as run:
    full_base_pre = WalmartBasePreprocessor()
    full_base_pre.fit(stores, features)
    train_full_all = full_base_pre.transform(train_raw)

    full_tab_eng = WalmartTabularFeatureEngineer()
    full_tab_eng.fit(train_full_all[["Store", "Dept", "Date", "Weekly_Sales"]])
    train_full_all = full_tab_eng.transform_train(train_full_all)

    X_train_all = train_full_all[feature_cols]
    y_train_all = train_full_all["Weekly_Sales"]

    final_model_full = lgb.LGBMRegressor(**best_params, random_state=42, n_jobs=-1)
    final_model_full.fit(X_train_all, y_train_all)

    mlflow.log_params(best_params)
    mlflow.log_param("training_data", "full_train_143_weeks")
    mlflow.log_metric("train_rows", len(train_full_all))

    final_pipeline = WalmartLGBMPipeline(
        base_preprocessor=full_base_pre,
        tabular_engineer=full_tab_eng,
        model=final_model_full,
        feature_cols=feature_cols,
    )

    mlflow.pyfunc.log_model(
        artifact_path="lightgbm_pipeline_full",
        python_model=final_pipeline,
        registered_model_name="Walmart_LightGBM_Pipeline",
    )

    print("Final model trained on full train.csv:", len(train_full_all), "rows")

2026/07/12 16:56:20 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009853 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4155
[LightGBM] [Info] Number of data points in the train set: 421570, number of used features: 31
[LightGBM] [Info] Start training from score 15981.258121
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

2026/07/12 16:56:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 16:56:40 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.
2026/07/12 16:56:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'Walmart_LightGBM_Pipeline' already exists. Creating a new version of this model...
2026/07/12 16:57:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_LightGBM_Pipeline, version 2
Created version '2' of model 'Walmart_LightGBM_Pipeline'.


Final model trained on full train.csv: 421570 rows
🏃 View run LightGBM_Final_Refit at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3/runs/cb8a98b569184729a25e828059909ea2
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/3
